# ML Forecasting Models

Trains direction-classification (up/down) and return-regression models using
Random Forest and XGBoost, then compares their test-set performance. Also fits
a sequence (LSTM-surrogate) model on a single ticker and an ARIMA baseline.

In [ ]:
import os
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

from src.models.supervised_ml import (
    build_features_and_target,
    get_feature_importances,
    train_random_forest,
    train_xgboost,
)
from src.models.time_series import (
    check_stationarity,
    fit_arima,
    forecast_arima,
    train_lstm,
)
from src.visualization.plots import (
    plot_feature_importances,
    plot_prediction_vs_actual,
)

%matplotlib inline
plt.rcParams['figure.dpi'] = 100

In [ ]:
returns = pd.read_csv('../data/raw/stock_returns.csv', index_col='Date')
returns.index = pd.to_datetime(returns.index)
returns = returns.select_dtypes(include='number').dropna()

prices = pd.read_csv('../data/raw/stock_data.csv', header=[0, 1], index_col=0)
prices.index = pd.to_datetime(prices.index)
# Extract Close prices (second level of multi-index columns)
close = prices['Close']

TARGET = 'AAPL'
print(f'Returns shape : {returns.shape}')
print(f'Close columns : {list(close.columns[:5])} ...')

## Direction Classification (up / down)

In [ ]:
X, y, feat_names = build_features_and_target(returns, ticker=TARGET, lags=5, target='direction')
print(f'Features: {X.shape[1]}, Samples: {X.shape[0]}')
print(f'Class balance: {np.bincount(y)}')

In [ ]:
rf_cls = train_random_forest(X, y, task='classification')
print('Random Forest accuracy:', rf_cls['metrics']['accuracy'])

In [ ]:
xgb_cls = train_xgboost(X, y, task='classification')
print('XGBoost accuracy:', xgb_cls['metrics']['accuracy'])

In [ ]:
importances = get_feature_importances(xgb_cls, feat_names)
fig = plot_feature_importances(importances, top_n=15, title=f'XGBoost Feature Importances ({TARGET} direction)')
plt.show()

## Return Regression

In [ ]:
X_r, y_r, feat_names_r = build_features_and_target(returns, ticker=TARGET, lags=5, target='return')

rf_reg = train_random_forest(X_r, y_r, task='regression')
print('RF regression metrics:', rf_reg['metrics'])

xgb_reg = train_xgboost(X_r, y_r, task='regression')
print('XGB regression metrics:', xgb_reg['metrics'])

## Sequence Model (Ridge on sliding windows)

In [ ]:
if TARGET in close.columns:
    ticker_prices = close[TARGET].dropna()
else:
    # Fallback: use first available column
    ticker_prices = close.iloc[:, 0].dropna()

lstm_result = train_lstm(ticker_prices, seq_length=60, test_size=0.2)
print('Sequence model metrics:', lstm_result['metrics'])

In [ ]:
fig = plot_prediction_vs_actual(
    lstm_result['y_test'], lstm_result['y_pred'],
    title=f'{TARGET} Price: Predicted vs Actual (sequence model)'
)
plt.show()

## ARIMA Baseline

In [ ]:
series = returns[TARGET].dropna()
stat   = check_stationarity(series)
print(f'ADF p-value: {stat["p_value"]:.4f}  stationary={stat["is_stationary"]}')

In [ ]:
# Train on first 80 %, forecast the next 30 steps and compare
split = int(len(series) * 0.8)
train_series = series.iloc[:split]
test_series  = series.iloc[split:split + 30]

fitted_arima  = fit_arima(train_series, order=(1, 0, 1))
arima_forecast = forecast_arima(fitted_arima, steps=30)

fig = plot_prediction_vs_actual(
    test_series.values, arima_forecast.values,
    title=f'{TARGET} Log-Returns: ARIMA(1,0,1) Forecast'
)
plt.show()